# M3 (Tracy) — Priority *direct predictor*

**Tracy's model.** Predict the priority directly from the summary plus
`plan_tier`, `customer_tenure_months` and `channel`. TF-IDF (stop-words removed,
1–2-grams, max 5000 features), class-balanced logistic regression. Evaluated on a
**chronological split** (for a fair head-to-head with Milind) and, for reference,
on Tracy's original **random stratified** split. Saves
`models/m3_priority_tracy.joblib`.

Note: this model keeps `plan_tier` and `tenure`, which Milind excludes as age
proxies — the comparison shows what that choice buys.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import northstar_prep as P
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score

df = P.load_clean()
FEATS, TARGET = ["case_summary","plan_tier","channel","customer_tenure_months"], "priority"

def tracy_pipe():
    trans = [("sum", TfidfVectorizer(max_features=5000, stop_words="english",
                                     ngram_range=(1,2), min_df=2, max_df=0.95), "case_summary"),
             P.t_cat(["plan_tier","channel"]),
             P.t_num(["customer_tenure_months"])]
    return P.make_clf(trans)

tr, va, te = P.time_split(df)
pipe = tracy_pipe().fit(tr[FEATS], tr[TARGET])
pred = pipe.predict(te[FEATS])
rank = P.PRIORITY_RANK
gap  = (te[TARGET].map(rank) - [rank[p] for p in pred]).abs()
metrics = {"macro_F1": round(f1_score(te[TARGET],pred,average="macro"),3),
           "accuracy": round(accuracy_score(te[TARGET],pred),3),
           "within_1_level": round((gap<=1).mean(),3),
           "baseline_acc": round(df[TARGET].value_counts(normalize=True).iloc[0],3)}

# reference: Tracy's original random stratified split
rtr, rte = train_test_split(df, test_size=0.2, random_state=42, stratify=df[TARGET])
rpred = tracy_pipe().fit(rtr[FEATS], rtr[TARGET]).predict(rte[FEATS])
metrics["macro_F1_random_split"] = round(f1_score(rte[TARGET],rpred,average="macro"),3)
metrics["accuracy_random_split"] = round(accuracy_score(rte[TARGET],rpred),3)
print(metrics)

In [ ]:
proba = pipe.predict_proba(te[FEATS])
test = te[["case_id","case_summary","plan_tier","channel","customer_tenure_months",TARGET]].copy().reset_index(drop=True)
test["predicted"], test["confidence"] = pred, proba.max(axis=1).round(3)
full = tracy_pipe().fit(df[FEATS], df[TARGET])
path = P.save_bundle("m3_priority_tracy", {
  "model": full, "target": TARGET, "features": FEATS, "classes": P.PRIORITIES,
  "kind":"multiclass", "label":"Priority (direct)", "metrics": metrics, "test": test,
  "meta":{"author":"Tracy","approach":"direct predictor","split":"time (+random ref)",
          "priorities":P.PRIORITIES,"inputs":"summary + plan_tier + tenure + channel"}})
print("saved", path)